# KAIROS, proof of concept

Five steps, in the order the pipeline runs them.

1. **Extract** a valve passport from the real de-identified notes and report aggregates only.
2. **Stage** an echocardiogram against the patient's own reference study under VARC-3.
3. **Build** a landmark dataset from a synthetic scenario.
4. **Fit** the cause-specific model and combine it into the reported probabilities.
5. **Compare** with current practice: guideline surveillance and the VARC-3 rule.

> Steps 3 to 5 run on **explicitly synthetic** scenarios. Every number they produce is evidence
> about the software, not about patients. No clinical accuracy is demonstrated or claimed.

Step 1 needs the three supplied spreadsheets at the repository root. They are gitignored and never
committed, so that step is skipped automatically when they are absent, and the rest still runs.

## 0. Setup


In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
pd.set_option('display.width', 120)

NOTES_XLSX = ROOT / 'notes_deidentified.xlsx'
HAVE_NOTES = NOTES_XLSX.exists()
print('real notes      :', 'present' if HAVE_NOTES else 'absent, step 1 will be skipped')

real notes      : absent, step 1 will be skipped


## 1. Extraction from the real notes

`prepare_notes` applies the deleted-status exclusion: 215 rows become 202 across 117 patients.


In [2]:
if HAVE_NOTES:
    from kairos.passport import load_notes, build_passport
    notes = load_notes()
    out = build_passport()
    passport = out[0] if isinstance(out, tuple) else out
    print(f'{len(notes)} notes, {notes["Profile Key"].nunique()} patients')
    print(f'{len(passport)} patient rows in the passport')
    print()
    print(passport['route'].value_counts(dropna=False).to_string())
else:
    print('skipped: the supplied spreadsheets are not in this checkout by design.')
    print('Committed aggregates derived from them:')
    agg = ROOT / 'data/derived/aggregates/patients_by_route.csv'
    print(agg.read_text() if agg.exists() else '  (aggregates not present)')


skipped: the supplied spreadsheets are not in this checkout by design.
Committed aggregates derived from them:
route,n_patients
SAVR,53
TAVR,44
(hist only),15
(unspecified),5



### The extraction trap that changed a published number

Transcatheter operative reports carry a structured field reading `Valve in Valve: No`. Matching the
phrase counts those patients as having had the event the field denies. The guard rejects any match
falling inside such a field. On this corpus the count goes from 10 patients to 4.

This runs without the spreadsheets, because the rule is demonstrated on two literal strings.


In [3]:
from kairos.passport import EVENT_PATS, negated_field_spans, in_negated_field

denied = 'PROCEDURE: TAVR\nValve in Valve: No\nAccess: transfemoral'
prose  = 'Underwent transfemoral TAVR (valve-in-valve) with a 26 mm Evolut FX.'

def fires(key, text):
    spans = negated_field_spans(text)
    return any(not in_negated_field(m.start(), spans) for m in EVENT_PATS[key].finditer(text))

print('phrase present in the denied field :', bool(EVENT_PATS['ViV'].search(denied)))
print('counted as an event                :', fires('ViV', denied))
print('genuine prose still counted        :', fires('ViV', prose))


phrase present in the denied field : True
counted as an event                : False
genuine prose still counted        : True


## 2. VARC-3 staging against the patient's own reference study

Change-based staging needs the patient's baseline. Published normal values give context and never
replace it. An input that cannot be resolved returns `uncertain`, never a negative.


In [4]:
from kairos.varc3 import Echo, stage_hvd

reference = Echo(mean_gradient_mmHg=11.0, dvi=0.48, eoa_cm2=1.70, regurg_grade=0)
year_four = Echo(mean_gradient_mmHg=19.0, dvi=0.37, eoa_cm2=1.25, regurg_grade=1)

result = stage_hvd(year_four, reference_echo=reference)
print('stage      :', result.stage)
print('reason     :', getattr(result, 'reason', getattr(result, 'rationale', '')))
print()
print(f'gradient rose {year_four.mean_gradient_mmHg - reference.mean_gradient_mmHg:.0f} mmHg.',
      'Stage 2 needs a rise of 10 to reach 20.')
print('This is the case the model exists to find: moving, but below every threshold.')


stage      : 1
reason     : Some worsening in gradient/EOA/DVI/regurgitation detected, but not meeting the stage-2 combined threshold. Stage 0/1 boundary is this module's own interpolation -- see module docstring; the full VARC-3 stage-1 definition also includes morphological (imaging) criteria not assessed here.

gradient rose 8 mmHg. Stage 2 needs a rise of 10 to reach 20.
This is the case the model exists to find: moving, but below every threshold.


## 3. A synthetic scenario and the landmark dataset

One row per patient per prediction time, every feature as known at that time. The builder enforces
the leakage rules: nothing dated after the landmark, patients who met the endpoint leave the risk
set, and all rows of one patient carry the same cluster id for resampling.


In [5]:
from kairos.simulation.scenarios import list_scenarios, get_scenario

for name, variant in list_scenarios():
    print(f'  {name}' + (f' / {variant}' if variant else ''))


  gradual_stenotic
  regurgitant_abrupt
  high_competing_mortality
  irregular_surveillance
  biomarker_information / meaningful
  biomarker_information / weak
  biomarker_information / absent
  biomarker_information / unmeasured
  anticoagulant_mechanism_confounding / marker_mediated
  anticoagulant_mechanism_confounding / marker_noise


In [6]:
from kairos.simulation.generators import generate_cohort
from kairos.modelling.landmark import build_landmark

spec   = get_scenario('gradual_stenotic')
cohort = generate_cohort(spec, seed=20260916)   # the scenario's 2,500 attempted patients
print('patients :', len(cohort.patients))
print('echoes   :', len(cohort.echoes))
print('events   :', len(cohort.events))

patients : 2390
echoes   : 11010
events   : 2390


In [7]:
build = build_landmark(
    patients=cohort.patients, echoes=cohort.echoes, labs=cohort.labs,
    exposures=cohort.exposures, events=cohort.events, horizon_years=5.0)
rows = build.rows
print('landmark rows      :', len(rows))
print('distinct patients  :', rows['patient_id'].nunique())
print('rows per patient   :', round(len(rows) / rows['patient_id'].nunique(), 1))
print('label policy       :', build.label_policy)
print('endpoint version   :', build.endpoint_version)
print()
print('exclusions applied :')
print(build.exclusions if isinstance(build.exclusions, str) else pd.Series(build.exclusions).to_string())


landmark rows      : 10701
distinct patients  : 2390
rows per patient   : 4.5
label policy       : primary
endpoint version   : 2

exclusions applied :
no_events_row                                  0
no_echo                                        0
no_reference                                   0
endpoint_at_or_before_reference                0
unresolved_candidate_at_or_before_reference    0
death_or_replacement_at_or_before_reference    0
followup_end_at_or_before_reference            0
no_eligible_landmark                           0


### The leakage guard is executable, not a promise

`truth_columns_in` raises if any column carrying simulated ground truth reaches the feature frame.


In [8]:
from kairos.modelling.landmark import truth_columns_in
leaked = truth_columns_in(rows)
print('truth columns present in the landmark rows:', leaked if leaked else 'none')
assert not leaked, 'simulated ground truth leaked into the features'
print('guard passed')


truth columns present in the landmark rows: none
guard passed


## 4. The model, and why it reports four probabilities

The `core` model is a penalised cause-specific Cox model: one hazard model each for structural
valve deterioration (SVD), death, and replacement of the valve for another reason, with baseline
hazards stratified by route. It is **trained** on the landmark rows (maximum partial likelihood,
ridge penalty) and **not tuned**: the penalty is fixed in `config/model.yaml`.

The three hazards are combined into cumulative incidence, so every prediction splits the future
into four states that add up to one: SVD first, death first, replacement for another reason
first, and alive with an intact valve. One minus the SVD risk is **not** the chance of being alive
with a working valve; it includes everyone who died first.

In [9]:
from kairos.modelling.modules import load_model_config
from kairos.modelling.train import fit_step
from kairos.evaluation.ladder import predict_states

cfg  = load_model_config()
core = next(s['blocks'] for s in cfg['ladder'] if s['name'] == 'core')
pipe, model, feats, *_ = fit_step(rows, core, cfg)
print('features used :', len(feats))
print('penalty       :', model.penalizer, '(fixed, not tuned)')
for cause, rec in model.support_.items():
    print(f'  {cause:12s} {rec["status"]:24s} {rec["event_patients"]} patients with the event')

features used : 41
penalty       : 0.05 (fixed, not tuned)
  svd          ok                       89 patients with the event
  death        ok                       1459 patients with the event
  replacement  ok                       33 patients with the event


Two landmarks from this cohort show why. For a very old transcatheter patient the SVD risk is tiny, but not because the valve is safe: death comes first in almost every future. A young surgical patient's valve is the one likely to wear out. A single SVD number, or one minus it read as "doing well", would hide both facts.

In [10]:
horizons = [float(h) for h in cfg['horizons_years']]
older   = rows[(rows['route'] == 'TAVR')].sort_values('age_at_implant').iloc[[-1]]
younger = rows[(rows['route'] == 'SAVR') & (rows['t_lm'] >= 3)].sort_values('age_at_implant').iloc[[0]]

for label, row in (('older TAVI patient', older), ('younger surgical patient', younger)):
    probs, _ = predict_states(pipe, model, row, horizons, 1.0)
    table = pd.DataFrame({s: probs[s][0] for s in ('svd', 'death', 'replacement', 'alive_intact')},
                         index=[f'{h:g} y' for h in horizons])
    table['sum'] = table.sum(axis=1)
    age = float(row['age_at_implant'].iloc[0]) + float(row['valve_age_years'].iloc[0])
    print(f'{label}: about {age:.0f} years old, valve {float(row["valve_age_years"].iloc[0]):.1f} years in place')
    print(table.round(3).to_string())
    print()

older TAVI patient: about 95 years old, valve 0.4 years in place
       svd  death  replacement  alive_intact  sum
1 y  0.001  0.494        0.001         0.505  1.0
3 y  0.002  0.886        0.003         0.110  1.0
5 y  0.002  0.974        0.003         0.021  1.0

younger surgical patient: about 49 years old, valve 4.4 years in place
       svd  death  replacement  alive_intact  sum
1 y  0.007  0.010        0.003         0.981  1.0
3 y  0.056  0.030        0.009         0.905  1.0
5 y  0.137  0.051        0.016         0.796  1.0



## 5. Against current practice

The VARC-3 rule reads the current echo against the reference study; the model predicts. Landmarks
stop before the first echo that meets the endpoint, so the rule is negative at every landmark by
construction, and the two are compared **over time** instead: each held-out patient's noise-free
valve trajectory is replayed under a guideline schedule, with and without KAIROS bringing the next
echo forward when the 12-month SVD risk is at least 2 percent. Guideline echoes are never removed,
and detection is the first echo on which the VARC-3 rule is positive.

`scripts/evaluate_surveillance.py` produces the full results (about a minute per scenario); this
cell reads the committed ones for the gradual stenotic scenario.

In [11]:
out = ROOT / 'docs' / 'comparison' / 'surveillance' / 'gradual_stenotic'
pol = pd.read_csv(out / 'policies.csv')
lead = pd.read_csv(out / 'lead_time.csv')
keep = ['annual', 'annual+kairos@0.02', 'acc_aha', 'acc_aha+kairos@0.02']
cols = ['policy', 'route', 'echoes_per_1000py', 'crossers', 'detected_pct', 'delay_median_months', 'delay_p90_months']
print(pol[pol['policy'].isin(keep) & pol['route'].isin(['all', 'SAVR'])][cols].to_string(index=False))
print()
print(lead[lead['threshold'] == 0.02][['policy', 'lead_mean_months', 'lead_mean_ci95', 'only_guided_detected',
                                         'only_base_detected', 'extra_echoes_per_1000py']].to_string(index=False))

             policy route  echoes_per_1000py  crossers  detected_pct  delay_median_months  delay_p90_months
             annual   all              886.3       182          61.5                  7.1              14.6
 annual+kairos@0.02   all              903.5       182          64.3                  6.2              10.5
            acc_aha   all              434.3       182          34.1                  9.9              21.8
acc_aha+kairos@0.02   all              442.5       182          44.5                  8.9              20.6
             annual  SAVR              901.9       148          63.5                  7.0              14.4
 annual+kairos@0.02  SAVR              930.2       148          67.6                  6.1              10.2
            acc_aha  SAVR              125.7       148          29.7                 10.9              23.3
acc_aha+kairos@0.02  SAVR              136.4       148          43.2                 10.2              20.5

             policy  lead_m

On a yearly schedule KAIROS catches deterioration about two months earlier on average for about 2
percent more echoes. On the ACC/AHA calendar, which images surgical valves at 5 and 10 years, it
raises the share of deteriorating surgical valves caught before follow-up ends from about 30 to 43
percent. The patients follow the generator's own model of deterioration, so this shows what the
model can do if that model is right; it is not clinical evidence. See
`docs/comparison/surveillance/README.md`.

## Reproducing everything

```bash
make test      # unit, contract and service tests
make quick     # scenarios, training and evaluation into artifacts/
```

Figures land in `artifacts/figures/`, the evaluation ladder in `artifacts/ladder_summary.md`.
`scripts/privacy_scan.py` runs in continuous integration and blocks any patient-level row from
reaching version control.
